
## RWF-2000 + UCF-Crime Fighting + Bus Violence

Notebook ini **TEST ONLY / EVALUATION ONLY**.

Tidak ada:
- training ulang;
- optimizer;
- scheduler;
- backpropagation;
- fine-tuning;
- model selection menggunakan held-out dataset.

Basis protokol mengikuti:
`EffTempNet_BiLSTMFinal_FrameDiversity_42.ipynb`.

Arsitektur dan frame-diversity protocol yang dipertahankan:

```text
shared 16-frame cache
→ pilih K distinct temporal positions
→ K ∈ {1,2,4,8,16}
→ untuk K<16: satu posisi reproducible dari tiap equal-width temporal bin
→ selected positions diulang secara chronological blocks sampai T kembali 16
→ EfficientFormerV2-S0 per frame
→ feature dropout
→ BiLSTM hidden_size=32, bidirectional
→ concat final forward/backward hidden states
→ classifier
→ clip prediction
```

Untuk:
- `K=1,2,4,8` → 10 temporal-sampling repeats;
- `K=16` → satu deterministic reference.

> **K=1 bukan model SingleFrame.**
>
> Checkpoint tetap **BiLSTM-Final T=16** dan sequence interface model tetap
> T=16. Yang dikurangi hanya jumlah distinct temporal observations, lalu
> selected observations diulang sampai panjang sequence kembali 16.

Karena BiLSTM adalah model sequential, pengulangan frame juga mengubah temporal
redundancy dan transition density. Karena itu eksperimen ini mengukur
**sensitivity to temporal diversity/redundancy at fixed T=16**, bukan isolasi
kausal murni jumlah informasi.

Held-out dataset dievaluasi **terpisah**:
1. RWF-2000
2. UCF-Crime Fighting
3. Bus Violence

Tidak ada pooling RWF + UCF + Bus.

## Mengganti seed

Setelah path dataset benar, **cukup ubah satu baris**:

```python
RUN_SEED = 0
```

menjadi `42` atau `123`.

In [ ]:
# Jalankan sekali pada awal sesi Colab.
!pip -q install timm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import gc
import json
import random
import platform
import warnings
import zlib

from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
)

import timm
from timm import create_model

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

print(
    "Python :",
    platform.python_version(),
)

print(
    "PyTorch:",
    torch.__version__,
)

print(
    "timm   :",
    timm.__version__,
)

print(
    "CUDA   :",
    torch.cuda.is_available(),
)

if torch.cuda.is_available():
    print(
        "GPU    :",
        torch.cuda.get_device_name(
            0
        ),
    )

Python : 3.13.15
PyTorch: 2.11.0+cu128
timm   : 1.0.28
CUDA   : True
GPU    : NVIDIA A100-SXM4-40GB


## 1. Konfigurasi

### Satu-satunya variabel yang diubah antar-seed

```python
RUN_SEED = 0
```

Valid:

```python
RUN_SEED ∈ {0,42,123}
```

Jangan ubah antar-seed:

```python
SEQUENCE_LENGTH = 16
LSTM_UNITS = 32
K_VALUES = [1,2,4,8,16]
NUM_RANDOM_REPEATS = 10
SAMPLING_BASE_SEED = 20260810
BATCH_SIZE = 4
```

`RUN_SEED` **tidak masuk** ke generator temporal sampling. Dengan demikian
checkpoint seed 0/42/123 menerima temporal selections yang sama untuk
dataset, sample, K, dan repeat yang sama.

In [ ]:
# ============================================================
# SATU-SATUNYA VARIABEL YANG DIUBAH ANTAR-SEED
# ============================================================
RUN_SEED = 0

VALID_SEEDS = (
    0,
    42,
    123,
)

if RUN_SEED not in VALID_SEEDS:
    raise ValueError(
        f"RUN_SEED harus salah satu dari {VALID_SEEDS}; "
        f"mendapat {RUN_SEED}."
    )

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# ============================================================
# EXACT BILSTM-FINAL PROTOCOL
# ============================================================
VARIANT = "bilstm_final"

NUM_CLASSES = 2

SEQUENCE_LENGTH = 16

RAW_HEIGHT = 64
RAW_WIDTH = 64

BACKBONE_INPUT_SIZE = 224

LSTM_UNITS = 32

K_VALUES = [
    1,
    2,
    4,
    8,
    16,
]

NUM_RANDOM_REPEATS = 10

# Must remain identical across checkpoint seeds.
SAMPLING_BASE_SEED = 20260810

BATCH_SIZE = 4

NUM_WORKERS = 2

PREFER_LEGACY_CACHE = True

LEGACY_CACHE_COLOR_ORDER = "BGR"

AUTO_SCALE_LEGACY_CACHE = True

ALLOW_BUILD_CACHE = True

# K=16 diversity route must reproduce direct original T=16 route.
K16_PROBABILITY_ATOL = 1e-6
K16_METRIC_ATOL = 1e-8

# ============================================================
# MODEL PATHS
# ============================================================
MODEL_ROOT = Path(
    "/content/drive/MyDrive/Models/"
)

CHECKPOINT_PATH_OVERRIDE = None


def resolve_checkpoint(
    seed: int,
) -> Path:
    if CHECKPOINT_PATH_OVERRIDE is not None:
        path = Path(
            CHECKPOINT_PATH_OVERRIDE
        )

        if not path.exists():
            raise FileNotFoundError(
                f"CHECKPOINT_PATH_OVERRIDE tidak ditemukan: {path}"
            )

        return path

    candidates = [
        (
            MODEL_ROOT
            / f"EffTempNet_BiLSTMFinal_NoLS_Seed{seed}_v1"
            / VARIANT
            / f"seed_{seed}"
            / "best.pt"
        ),
        (
            MODEL_ROOT
            / f"EffTempNet_BiLSTMFinal_NoLS_{seed}Seeds_v1"
            / VARIANT
            / f"seed_{seed}"
            / "best.pt"
        ),
        (
            MODEL_ROOT
            / f"EffTempNet_BiLSTMFinal_NoLS_{seed}Seed_v1"
            / VARIANT
            / f"seed_{seed}"
            / "best.pt"
        ),
    ]

    matches = [
        path
        for path in candidates
        if path.exists()
    ]

    matches = list(
        dict.fromkeys(
            matches
        )
    )

    if len(
        matches
    ) == 1:
        return matches[
            0
        ]

    if len(
        matches
    ) > 1:
        raise RuntimeError(
            "Ditemukan beberapa checkpoint BiLSTM-Final. "
            "Set CHECKPOINT_PATH_OVERRIDE secara eksplisit:\n"
            + "\n".join(
                str(
                    path
                )
                for path
                in matches
            )
        )

    recursive = sorted(
        MODEL_ROOT.glob(
            f"**/{VARIANT}/seed_{seed}/best.pt"
        )
    )

    # Only paths clearly belonging to BiLSTMFinal.
    recursive = [
        path
        for path
        in recursive
        if (
            "bilstmfinal"
            in str(
                path
            )
            .lower()
            .replace(
                "_",
                "",
            )
        )
    ]

    if len(
        recursive
    ) == 1:
        return recursive[
            0
        ]

    if len(
        recursive
    ) == 0:
        raise FileNotFoundError(
            "Checkpoint BiLSTM-Final tidak ditemukan.\n"
            f"Seed={seed}\n"
            f"Search root={MODEL_ROOT}\n"
            "Jika struktur folder berbeda, isi CHECKPOINT_PATH_OVERRIDE."
        )

    raise RuntimeError(
        "Ditemukan beberapa checkpoint BiLSTM-Final yang ambigu. "
        "Set CHECKPOINT_PATH_OVERRIDE:\n"
        + "\n".join(
            str(
                path
            )
            for path
            in recursive
        )
    )


CHECKPOINT_PATH = resolve_checkpoint(
    RUN_SEED
)

# ============================================================
# HELD-OUT DATA PATHS
# ============================================================
BENCHMARK_ROOT = Path(
    "/content/drive/MyDrive/Benchmark"
)

EXISTING_CACHE_DIR = (
    MODEL_ROOT
    / "new_saved_dataset"
)

HELDOUT_CACHE_DIR = (
    MODEL_ROOT
    / "heldout_cache_rgb64_v2"
)

HELDOUT_SPECS = {
    "RWF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "RWF"
        ),
        "cache_name": "RWF",
        "cache_dir": EXISTING_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },

    "UCF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "UCF"
        ),
        "cache_name": "UCF",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },

    "BV": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "BV"
        ),
        "cache_name": "BV",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
}

NONVIOLENCE_ALIASES = (
    "NonViolence",
    "Non-Violence",
    "Non_Violence",
    "Nonviolent",
    "NonViolent",
    "NonFight",
    "Non-Fight",
    "Non_Fight",
    "Normal",
    "NormalVideos",
    "Normal_Videos",
)

VIOLENCE_ALIASES = (
    "Violence",
    "Violent",
    "Fight",
    "Fighting",
    "FightVideos",
    "ViolentVideos",
    "Abnormal",
)

EVAL_OUTPUT_DIR = (
    MODEL_ROOT
    / "HeldOut_Evaluation"
    / "BiLSTMFinal_FrameDiversity"
    / f"seed_{RUN_SEED}"
)

EVAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HELDOUT_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for k_unique in K_VALUES:
    if (
        SEQUENCE_LENGTH
        % k_unique
        != 0
    ):
        raise ValueError(
            f"K={k_unique} tidak membagi T={SEQUENCE_LENGTH}."
        )

print(
    "RUN_SEED      :",
    RUN_SEED,
)

print(
    "DEVICE        :",
    DEVICE,
)

print(
    "CHECKPOINT    :",
    CHECKPOINT_PATH,
)

print(
    "K values      :",
    K_VALUES,
)

print(
    "Repeats       :",
    NUM_RANDOM_REPEATS,
)

print(
    "Sampling seed :",
    SAMPLING_BASE_SEED,
)

print(
    "OUTPUT        :",
    EVAL_OUTPUT_DIR,
)

RUN_SEED      : 0
DEVICE        : cuda
CHECKPOINT    : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/EffTempNet_BiLSTMFinal_NoLS_Seed0_v1/bilstm_final/seed_0/best.pt
K values      : [1, 2, 4, 8, 16]
Repeats       : 10
Sampling seed : 20260810
OUTPUT        : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_FrameDiversity/seed_0


In [ ]:
def set_eval_seed(
    seed: int,
):
    os.environ[
        "PYTHONHASHSEED"
    ] = str(
        seed
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(
    worker_id: int,
):
    worker_seed = (
        torch.initial_seed()
        % (2**32)
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


def safe_torch_load(
    path: Path,
    map_location,
):
    path = Path(
        path
    )

    load_path = path

    if str(
        path
    ).startswith(
        "/content/drive/"
    ):
        local_cache_dir = Path(
            "/content/_checkpoint_staging"
        )

        local_cache_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        path_crc = (
            zlib.crc32(
                str(
                    path
                ).encode(
                    "utf-8"
                )
            )
            & 0xFFFFFFFF
        )

        local_path = (
            local_cache_dir
            / f"{path_crc:08x}_{path.name}"
        )

        partial_path = (
            local_path.with_suffix(
                local_path.suffix
                + ".part"
            )
        )

        def stage_checkpoint():
            expected_size = (
                path.stat().st_size
            )

            if partial_path.exists():
                partial_path.unlink()

            with open(
                path,
                "rb",
            ) as source_file:
                with open(
                    partial_path,
                    "wb",
                ) as destination_file:
                    while True:
                        chunk = source_file.read(
                            8
                            * 1024
                            * 1024
                        )

                        if not chunk:
                            break

                        destination_file.write(
                            chunk
                        )

            copied_size = (
                partial_path.stat().st_size
            )

            if (
                copied_size
                != expected_size
            ):
                raise OSError(
                    "Ukuran checkpoint lokal tidak sama dengan sumber: "
                    f"{copied_size} != {expected_size}"
                )

            os.replace(
                partial_path,
                local_path,
            )

            return local_path

        try:
            load_path = stage_checkpoint()

        except OSError as error:
            is_transport_error = (
                getattr(
                    error,
                    "errno",
                    None,
                )
                == 107
                or "Transport endpoint is not connected"
                in str(
                    error
                )
            )

            if not is_transport_error:
                raise

            if partial_path.exists():
                partial_path.unlink()

            from google.colab import drive as colab_drive

            try:
                colab_drive.flush_and_unmount()

            except Exception:
                pass

            colab_drive.mount(
                "/content/drive",
                force_remount=True,
            )

            load_path = stage_checkpoint()

    try:
        return torch.load(
            load_path,
            map_location=map_location,
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            load_path,
            map_location=map_location,
        )


def assert_finite_tensor(
    name: str,
    tensor: torch.Tensor,
):
    if not torch.isfinite(
        tensor
    ).all():
        raise RuntimeError(
            f"{name} mengandung NaN/Inf."
        )


set_eval_seed(
    RUN_SEED
)

## 2. Exact BiLSTM-Final architecture

Sama dengan notebook source:

```text
16 frames
→ EfficientFormerV2-S0 shared per frame
→ feature dropout 0.2
→ 1-layer bidirectional LSTM, hidden_size=32
→ h_n[-2] forward-final
→ h_n[-1] backward-final
→ concatenate → 64-D clip context
→ classifier 64→256→128→64→32→2
```

Tidak ada temporal attention.

In [ ]:
class EffTempNetBiLSTMFinal(
    nn.Module
):
    VALID_VARIANTS = {
        "bilstm_final"
    }

    def __init__(
        self,
        variant: str = "bilstm_final",
        backbone_name: str = "efficientformerv2_s0",
        num_classes: int = NUM_CLASSES,
        lstm_units: int = LSTM_UNITS,
        pretrained: bool = False,
    ):
        super().__init__()

        if variant not in self.VALID_VARIANTS:
            raise ValueError(
                f"Unknown variant: {variant}"
            )

        self.variant = variant

        self.backbone = create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0,
            in_chans=3,
        )

        self.feature_dim = int(
            self.backbone.num_features
        )

        self.feature_dropout = nn.Dropout(
            0.2
        )

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=lstm_units,
            batch_first=True,
            bidirectional=True,
        )

        self.temporal_dim = (
            2
            * lstm_units
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                self.temporal_dim,
                256,
            ),
            nn.ReLU(),
            nn.Dropout(
                0.25
            ),

            nn.Linear(
                256,
                128,
            ),
            nn.ReLU(),
            nn.Dropout(
                0.25
            ),

            nn.Linear(
                128,
                64,
            ),
            nn.ReLU(),
            nn.Dropout(
                0.2
            ),

            nn.Linear(
                64,
                32,
            ),
            nn.ReLU(),
            nn.Dropout(
                0.2
            ),

            nn.Linear(
                32,
                num_classes,
            ),
        )

        self._initialize_classifier()

        self.attention = None

    def _initialize_classifier(
        self,
    ):
        for module in self.classifier.modules():
            if isinstance(
                module,
                nn.Linear,
            ):
                nn.init.kaiming_uniform_(
                    module.weight,
                    nonlinearity="relu",
                )

                if module.bias is not None:
                    nn.init.zeros_(
                        module.bias
                    )

    def forward(
        self,
        clip: torch.Tensor,
        return_attention: bool = False,
    ):
        if clip.ndim != 5:
            raise ValueError(
                f"Expected [B,T,C,H,W], got {tuple(clip.shape)}"
            )

        (
            batch_size,
            seq_len,
            channels,
            height,
            width,
        ) = clip.shape

        if (
            seq_len
            != SEQUENCE_LENGTH
        ):
            raise ValueError(
                f"Expected T={SEQUENCE_LENGTH}, got T={seq_len}"
            )

        if channels != 3:
            raise ValueError(
                f"Expected RGB channels=3, got {channels}"
            )

        x = clip.reshape(
            batch_size
            * seq_len,
            channels,
            height,
            width,
        )

        x = F.interpolate(
            x,
            size=(
                BACKBONE_INPUT_SIZE,
                BACKBONE_INPUT_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        features = self.backbone(
            x
        )

        if features.ndim == 4:
            features = (
                F.adaptive_avg_pool2d(
                    features,
                    output_size=1,
                )
                .flatten(
                    1
                )
            )

        elif features.ndim != 2:
            features = features.flatten(
                1
            )

        features = features.view(
            batch_size,
            seq_len,
            -1,
        )

        features = self.feature_dropout(
            features
        )

        (
            _,
            (
                h_n,
                _,
            ),
        ) = self.lstm(
            features
        )

        context = torch.cat(
            [
                h_n[
                    -2
                ],
                h_n[
                    -1
                ],
            ],
            dim=1,
        )

        logits = self.classifier(
            context
        )

        if return_attention:
            return (
                logits,
                None,
            )

        return logits


# ============================================================
# STRICT CHECKPOINT LOAD
# ============================================================
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint tidak ditemukan: {CHECKPOINT_PATH}"
    )

checkpoint = safe_torch_load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

if not isinstance(
    checkpoint,
    dict,
):
    raise TypeError(
        "Checkpoint bukan dictionary."
    )

if (
    "model_state_dict"
    not in checkpoint
):
    raise KeyError(
        "Checkpoint tidak mempunyai model_state_dict."
    )

checkpoint_config = checkpoint.get(
    "config",
    {},
)

if "variant" in checkpoint_config:
    if str(
        checkpoint_config[
            "variant"
        ]
    ) != VARIANT:
        raise RuntimeError(
            "Checkpoint variant mismatch."
        )

if "seed" in checkpoint_config:
    if int(
        checkpoint_config[
            "seed"
        ]
    ) != RUN_SEED:
        raise RuntimeError(
            "Checkpoint seed mismatch."
        )

if "sequence_length" in checkpoint_config:
    if int(
        checkpoint_config[
            "sequence_length"
        ]
    ) != SEQUENCE_LENGTH:
        raise RuntimeError(
            "Checkpoint sequence_length mismatch."
        )

model = EffTempNetBiLSTMFinal(
    variant=VARIANT,
    lstm_units=LSTM_UNITS,
    pretrained=False,
).to(
    DEVICE
)

load_result = model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

model.eval()

for parameter in model.parameters():
    parameter.requires_grad_(
        False
    )

if model.lstm is None:
    raise RuntimeError(
        "BiLSTM-Final harus mempunyai LSTM."
    )

if not model.lstm.bidirectional:
    raise RuntimeError(
        "BiLSTM-Final harus bidirectional."
    )

if (
    model.lstm.hidden_size
    != LSTM_UNITS
):
    raise RuntimeError(
        "BiLSTM hidden_size mismatch."
    )

if model.attention is not None:
    raise RuntimeError(
        "BiLSTM-Final tidak menggunakan temporal attention."
    )

with torch.inference_mode():
    dummy = torch.zeros(
        2,
        SEQUENCE_LENGTH,
        3,
        RAW_HEIGHT,
        RAW_WIDTH,
        device=DEVICE,
    )

    dummy_logits = model(
        dummy
    )

if (
    tuple(
        dummy_logits.shape
    )
    != (
        2,
        NUM_CLASSES,
    )
):
    raise RuntimeError(
        f"Unexpected output shape: "
        f"{tuple(dummy_logits.shape)}"
    )

assert_finite_tensor(
    "dummy_logits",
    dummy_logits,
)

total_params = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)

print(
    "✅ Strict BiLSTM-Final checkpoint/model audit PASS"
)

print(
    "Epoch      :",
    checkpoint.get(
        "epoch",
        "unknown",
    ),
)

print(
    "Best score :",
    checkpoint.get(
        "best_score",
        "unknown",
    ),
)

print(
    "LSTM units :",
    model.lstm.hidden_size,
)

print(
    "Bidirect.  :",
    model.lstm.bidirectional,
)

print(
    "Parameters :",
    f"{total_params:,}",
)

print(
    "Load result:",
    load_result,
)

del (
    dummy,
    dummy_logits,
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

✅ Strict BiLSTM-Final checkpoint/model audit PASS
Epoch      : 14
Best score : 0.8812722919822154
LSTM units : 32
Bidirect.  : True
Parameters : 3,359,954
Load result: <All keys matched successfully>


## 3. Resolve / build shared held-out 16-frame cache

RWF menggunakan cache eksperimen yang sudah ada bila tersedia.

UCF dan Bus Violence menggunakan shared held-out RGB-v2 cache yang sama dengan
evaluator sebelumnya.

Raw-video cache **tidak bergantung pada checkpoint seed**.

In [ ]:
# ============================================================
# SELF-CONTAINED CACHE-BUILDER CONSTANTS
# ============================================================
# Penting:
# Default argument Python dievaluasi pada SAAT function didefinisikan.
# Karena itu cache builder tidak boleh bergantung pada SEQUENCE_LENGTH /
# RAW_HEIGHT / RAW_WIDTH yang mungkin belum ada bila cell dijalankan ulang
# secara terpisah setelah runtime restart.
#
# Nilai ini adalah bagian tetap dari held-out cache protocol:
# T_cache=16, raw=64x64.
_CACHE_BUILDER_SEQUENCE_LENGTH = 16
_CACHE_BUILDER_RAW_HEIGHT = 64
_CACHE_BUILDER_RAW_WIDTH = 64

# Jika configuration cell sudah dijalankan, pastikan nilainya konsisten.
if (
    "SEQUENCE_LENGTH" in globals()
    and int(globals()["SEQUENCE_LENGTH"])
    != _CACHE_BUILDER_SEQUENCE_LENGTH
):
    raise RuntimeError(
        "Protocol mismatch: SEQUENCE_LENGTH harus 16."
    )

if (
    "RAW_HEIGHT" in globals()
    and int(globals()["RAW_HEIGHT"])
    != _CACHE_BUILDER_RAW_HEIGHT
):
    raise RuntimeError(
        "Protocol mismatch: RAW_HEIGHT harus 64."
    )

if (
    "RAW_WIDTH" in globals()
    and int(globals()["RAW_WIDTH"])
    != _CACHE_BUILDER_RAW_WIDTH
):
    raise RuntimeError(
        "Protocol mismatch: RAW_WIDTH harus 64."
    )

VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv",
    ".mpeg", ".mpg", ".webm"
}

def direct_subdirs(root: Path) -> Dict[str, Path]:
    if not root.exists():
        return {}
    return {
        p.name: p
        for p in root.iterdir()
        if p.is_dir()
    }

def resolve_one_class_dir(
    root: Path,
    explicit_name: Optional[str],
    aliases: Tuple[str, ...],
    canonical_name: str,
) -> Path:
    if explicit_name is not None:
        candidate = root / explicit_name
        if not candidate.exists():
            raise FileNotFoundError(
                f"Folder {canonical_name} tidak ditemukan: {candidate}"
            )
        return candidate

    subdirs = direct_subdirs(root)
    lower_map = {
        name.lower(): path
        for name, path in subdirs.items()
    }

    matches = []
    for alias in aliases:
        candidate = lower_map.get(alias.lower())
        if candidate is not None and candidate not in matches:
            matches.append(candidate)

    if len(matches) != 1:
        raise RuntimeError(
            f"Tidak dapat menentukan folder kelas {canonical_name} "
            f"secara aman pada {root}.\n"
            f"Matches: {[p.name for p in matches]}\n"
            f"Available: {sorted(subdirs.keys())}\n"
            "Isi nonviolence_dir/violence_dir di HELDOUT_SPECS."
        )

    return matches[0]

def list_videos_for_spec(
    raw_dir: Path,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
) -> List[Tuple[Path, int]]:
    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Raw held-out directory tidak ditemukan: {raw_dir}"
        )

    class_dirs = {
        0: resolve_one_class_dir(
            raw_dir,
            nonviolence_dir,
            NONVIOLENCE_ALIASES,
            "NonViolence",
        ),
        1: resolve_one_class_dir(
            raw_dir,
            violence_dir,
            VIOLENCE_ALIASES,
            "Violence",
        ),
    }

    records = []

    for label, class_dir in class_dirs.items():
        videos = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        )

        if not videos:
            raise RuntimeError(
                f"Tidak ada video pada folder kelas: {class_dir}"
            )

        print(
            f"{CLASS_NAMES[label]:12s}: {len(videos):5d} clips | {class_dir}"
        )

        records.extend(
            (p, label)
            for p in videos
        )

    labels = np.asarray(
        [label for _, label in records],
        dtype=np.int64,
    )

    if set(np.unique(labels).tolist()) != {0, 1}:
        raise RuntimeError(
            "Held-out dataset harus mempunyai kedua kelas: "
            "NonViolence dan Violence."
        )

    return records

def extract_uniform_rgb_frames(
    video_path: Path,
    sequence_length: int = _CACHE_BUILDER_SEQUENCE_LENGTH,
    height: int = _CACHE_BUILDER_RAW_HEIGHT,
    width: int = _CACHE_BUILDER_RAW_WIDTH,
) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if frame_count <= 0:
        cap.release()
        return None

    frame_positions = np.linspace(
        0,
        max(frame_count - 1, 0),
        sequence_length,
    ).astype(int)

    frames = []

    for pos in frame_positions:
        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(pos),
        )

        ok, frame_bgr = cap.read()

        if not ok:
            break

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )

        frame_rgb = cv2.resize(
            frame_rgb,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        )

        frames.append(
            frame_rgb.astype(np.float32) / 255.0
        )

    cap.release()

    if not frames:
        return None

    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    return np.stack(
        frames[:sequence_length],
        axis=0,
    ).astype(np.float32)

def build_rgb_cache(
    raw_dir: Path,
    cache_dir: Path,
    cache_name: str,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    feature_path = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    label_path = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    paths_path = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    records = list_videos_for_spec(
        raw_dir,
        nonviolence_dir,
        violence_dir,
    )

    # Gunakan temporary memmap agar tidak menyimpan semua clip di RAM.
    n_requested = len(records)

    temp_path = cache_dir / f".{cache_name}_tmp_features.npy"

    if temp_path.exists():
        temp_path.unlink()

    temp = np.lib.format.open_memmap(
        temp_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            n_requested,
            _CACHE_BUILDER_SEQUENCE_LENGTH,
            _CACHE_BUILDER_RAW_HEIGHT,
            _CACHE_BUILDER_RAW_WIDTH,
            3,
        ),
    )

    labels_buffer = np.empty(
        n_requested,
        dtype=np.int64,
    )

    valid_paths = []
    failed_paths = []
    valid_count = 0

    for video_path, label in tqdm(
        records,
        desc=f"Build cache {cache_name}",
        unit="video",
    ):
        clip = extract_uniform_rgb_frames(video_path)

        if clip is None:
            failed_paths.append(str(video_path))
            continue

        temp[valid_count] = clip
        labels_buffer[valid_count] = int(label)
        valid_paths.append(str(video_path))
        valid_count += 1

    temp.flush()
    del temp

    if valid_count == 0:
        if temp_path.exists():
            temp_path.unlink()
        raise RuntimeError(
            f"Semua video gagal didekode untuk {cache_name}."
        )

    # Compact hanya sample valid ke final .npy.
    temp_read = np.load(
        temp_path,
        mmap_mode="r",
    )

    final = np.lib.format.open_memmap(
        feature_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            valid_count,
            _CACHE_BUILDER_SEQUENCE_LENGTH,
            _CACHE_BUILDER_RAW_HEIGHT,
            _CACHE_BUILDER_RAW_WIDTH,
            3,
        ),
    )

    chunk_size = 16

    for start in range(0, valid_count, chunk_size):
        end = min(start + chunk_size, valid_count)
        final[start:end] = temp_read[start:end]

    final.flush()
    del final, temp_read

    np.save(
        label_path,
        labels_buffer[:valid_count],
    )

    with open(
        paths_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "paths": valid_paths,
                "failed": failed_paths,
                "n_requested": n_requested,
                "n_valid": valid_count,
                "color_order": "RGB",
                "sequence_length": _CACHE_BUILDER_SEQUENCE_LENGTH,
                "height": _CACHE_BUILDER_RAW_HEIGHT,
                "width": _CACHE_BUILDER_RAW_WIDTH,
            },
            f,
            indent=2,
        )

    if temp_path.exists():
        temp_path.unlink()

    print(
        f"Saved {feature_path} | "
        f"shape=({valid_count},{_CACHE_BUILDER_SEQUENCE_LENGTH},{_CACHE_BUILDER_RAW_HEIGHT},{_CACHE_BUILDER_RAW_WIDTH},3)"
    )
    print("Failed videos:", len(failed_paths))

    return {
        "feature_path": feature_path,
        "label_path": label_path,
        "paths_path": paths_path,
        "color_order": "RGB",
        "cache_source": "built_rgb64_v2",
    }

def resolve_heldout_store(
    dataset_name: str,
    spec: Dict,
) -> Dict:
    raw_dir = Path(spec["raw_dir"])
    cache_dir = Path(spec["cache_dir"])
    cache_name = str(spec["cache_name"])

    rgb_feature = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    rgb_label = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    rgb_paths = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    legacy_feature = cache_dir / f"{cache_name}_features.npy"
    legacy_label = cache_dir / f"{cache_name}_labels.npy"
    legacy_paths = cache_dir / f"{cache_name}_paths.json"

    # Sama dengan training: RGB-v2 diprioritaskan; legacy dipakai jika v2 tidak ada.
    if rgb_feature.exists() and rgb_label.exists():
        print(f"✅ {dataset_name}: RGB-v2 cache")
        return {
            "feature_path": rgb_feature,
            "label_path": rgb_label,
            "paths_path": rgb_paths if rgb_paths.exists() else None,
            "color_order": "RGB",
            "cache_source": "rgb64_v2",
        }

    if (
        PREFER_LEGACY_CACHE
        and legacy_feature.exists()
        and legacy_label.exists()
    ):
        print(
            f"✅ {dataset_name}: legacy cache "
            f"({LEGACY_CACHE_COLOR_ORDER}→RGB saat loading)"
        )
        return {
            "feature_path": legacy_feature,
            "label_path": legacy_label,
            "paths_path": legacy_paths if legacy_paths.exists() else None,
            "color_order": LEGACY_CACHE_COLOR_ORDER,
            "cache_source": "legacy",
        }

    if not ALLOW_BUILD_CACHE:
        raise FileNotFoundError(
            f"Cache {dataset_name} tidak ada dan ALLOW_BUILD_CACHE=False."
        )

    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Cache dan raw folder tidak ditemukan untuk {dataset_name}.\n"
            f"Raw expected: {raw_dir}"
        )

    print(
        f"ℹ️ {dataset_name}: cache belum ada, dibuat dari {raw_dir}"
    )

    return build_rgb_cache(
        raw_dir=raw_dir,
        cache_dir=cache_dir,
        cache_name=cache_name,
        nonviolence_dir=spec.get("nonviolence_dir"),
        violence_dir=spec.get("violence_dir"),
    )

## 4. Frame-diversity sampling

Untuk K<16:
1. 16 temporal positions dibagi menjadi K equal-width bins;
2. satu position dipilih secara reproducible dari tiap bin;
3. selected positions tetap chronological;
4. selected positions diulang secara equal-weight hingga sequence kembali T=16.

Contoh K=4:

```text
distinct positions : [2,6,10,14]
expanded T=16      : [2,2,2,2,6,6,6,6,10,10,10,10,14,14,14,14]
```

Karena model sequential, repeated blocks memang memengaruhi transition pattern
yang dilihat BiLSTM dan merupakan bagian dari perturbasi yang diuji.

In [ ]:
IMAGENET_MEAN = torch.tensor(
    [
        0.485,
        0.456,
        0.406,
    ],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)

IMAGENET_STD = torch.tensor(
    [
        0.229,
        0.224,
        0.225,
    ],
    dtype=torch.float32,
).view(
    1,
    3,
    1,
    1,
)


def stable_dataset_code(
    dataset_name: str,
) -> int:
    return int(
        zlib.crc32(
            dataset_name.encode(
                "utf-8"
            )
        )
        & 0xFFFFFFFF
    )


def stratified_unique_indices(
    dataset_name: str,
    source_idx: int,
    k_unique: int,
    repeat_id: int,
) -> np.ndarray:
    k_unique = int(
        k_unique
    )

    if k_unique not in K_VALUES:
        raise ValueError(
            f"K tidak valid: {k_unique}"
        )

    if (
        k_unique
        == SEQUENCE_LENGTH
    ):
        return np.arange(
            SEQUENCE_LENGTH,
            dtype=np.int64,
        )

    bin_width = (
        SEQUENCE_LENGTH
        // k_unique
    )

    seed_sequence = np.random.SeedSequence(
        [
            int(
                SAMPLING_BASE_SEED
            ),
            int(
                repeat_id
            ),
            int(
                stable_dataset_code(
                    dataset_name
                )
            ),
            int(
                source_idx
            ),
            int(
                k_unique
            ),
        ]
    )

    rng = np.random.default_rng(
        seed_sequence
    )

    selected = []

    for bin_id in range(
        k_unique
    ):
        start = (
            bin_id
            * bin_width
        )

        stop = (
            start
            + bin_width
        )

        selected.append(
            int(
                rng.integers(
                    start,
                    stop,
                )
            )
        )

    selected = np.asarray(
        selected,
        dtype=np.int64,
    )

    if (
        len(
            np.unique(
                selected
            )
        )
        != k_unique
    ):
        raise RuntimeError(
            "Jumlah distinct positions tidak sama dengan K."
        )

    if not np.all(
        np.diff(
            selected
        )
        > 0
    ):
        raise RuntimeError(
            "Distinct positions harus tetap chronological."
        )

    return selected


def expand_equal_weight_to_t16(
    unique_indices: np.ndarray,
) -> np.ndarray:
    unique_indices = np.asarray(
        unique_indices,
        dtype=np.int64,
    )

    repeat_factor = (
        SEQUENCE_LENGTH
        // len(
            unique_indices
        )
    )

    expanded = np.repeat(
        unique_indices,
        repeat_factor,
    ).astype(
        np.int64
    )

    if (
        len(
            expanded
        )
        != SEQUENCE_LENGTH
    ):
        raise RuntimeError(
            "Expanded sequence harus T=16."
        )

    counts = np.bincount(
        expanded,
        minlength=SEQUENCE_LENGTH,
    )

    nonzero_counts = counts[
        counts
        > 0
    ]

    if (
        len(
            np.unique(
                nonzero_counts
            )
        )
        != 1
    ):
        raise RuntimeError(
            "Equal-weight expansion gagal."
        )

    return expanded


# ------------------------------------------------------------
# Determinism / protocol audit
# ------------------------------------------------------------
audit_rows = []

for k_unique in K_VALUES:
    repeat_ids = (
        [0]
        if k_unique == 16
        else range(
            3
        )
    )

    for repeat_id in repeat_ids:
        first = (
            stratified_unique_indices(
                dataset_name="RWF",
                source_idx=123,
                k_unique=k_unique,
                repeat_id=repeat_id,
            )
        )

        second = (
            stratified_unique_indices(
                dataset_name="RWF",
                source_idx=123,
                k_unique=k_unique,
                repeat_id=repeat_id,
            )
        )

        if not np.array_equal(
            first,
            second,
        ):
            raise RuntimeError(
                "Frame sampling tidak deterministic."
            )

        expanded = (
            expand_equal_weight_to_t16(
                first
            )
        )

        audit_rows.append(
            {
                "k_unique": k_unique,
                "repeat": repeat_id,
                "distinct_indices_0based": first.tolist(),
                "expanded_indices_0based": expanded.tolist(),
            }
        )

frame_selection_audit_df = pd.DataFrame(
    audit_rows
)

frame_selection_audit_df.to_json(
    EVAL_OUTPUT_DIR
    / "frame_selection_audit_examples.json",
    orient="records",
    indent=2,
)

display(
    frame_selection_audit_df
)

print(
    "✅ Frame-selection generator audit PASS"
)

,k_unique,repeat,distinct_indices_0based,expanded_indices_0based
0,1,0,[15],"[15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 1..."
1,1,1,[8],"[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]"
2,1,2,[1],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
3,2,0,"[6, 12]","[6, 6, 6, 6, 6, 6, 6, 6, 12, 12, 12, 12, 12, 1..."
4,2,1,"[5, 11]","[5, 5, 5, 5, 5, 5, 5, 5, 11, 11, 11, 11, 11, 1..."
5,2,2,"[4, 9]","[4, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 9, 9, 9]"
6,4,0,"[0, 5, 10, 14]","[0, 0, 0, 0, 5, 5, 5, 5, 10, 10, 10, 10, 14, 1..."
7,4,1,"[1, 6, 10, 14]","[1, 1, 1, 1, 6, 6, 6, 6, 10, 10, 10, 10, 14, 1..."
8,4,2,"[3, 5, 8, 12]","[3, 3, 3, 3, 5, 5, 5, 5, 8, 8, 8, 8, 12, 12, 1..."
9,8,0,"[1, 2, 4, 7, 8, 10, 13, 14]","[1, 1, 2, 2, 4, 4, 7, 7, 8, 8, 10, 10, 13, 13,..."


✅ Frame-selection generator audit PASS


## 5. Direct-T16 dan Frame-Diversity datasets

Dua jalur digunakan:

- `DirectT16`: memakai semua 16 cached positions secara langsung;
- `FrameDiversity`: memakai K distinct positions lalu equal-weight repetition
  kembali ke T=16.

K=16 FrameDiversity harus mereproduksi DirectT16 sebelum K<16 dijalankan.

In [ ]:
class HeldOut16CacheBase(
    Dataset
):
    def __init__(
        self,
        dataset_name: str,
        feature_path: Path,
        label_path: Path,
        color_order: str,
    ):
        self.dataset_name = str(
            dataset_name
        )

        self.features = np.load(
            feature_path,
            mmap_mode="r",
        )

        self.labels = np.load(
            label_path,
            mmap_mode="r",
        ).astype(
            np.int64
        )

        self.color_order = str(
            color_order
        ).upper()

        if (
            len(
                self.features
            )
            != len(
                self.labels
            )
        ):
            raise ValueError(
                f"{dataset_name}: feature/label length mismatch."
            )

        expected_shape = (
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        )

        if (
            self.features.ndim
            != 5
            or tuple(
                self.features.shape[
                    1:
                ]
            )
            != expected_shape
        ):
            raise ValueError(
                f"{dataset_name}: cache shape={self.features.shape}; "
                f"expected [N,{SEQUENCE_LENGTH},"
                f"{RAW_HEIGHT},{RAW_WIDTH},3]."
            )

        unique_labels = set(
            np.unique(
                self.labels
            ).tolist()
        )

        if (
            unique_labels
            != {
                0,
                1,
            }
        ):
            raise ValueError(
                f"{dataset_name}: held-out labels harus {{0,1}}, "
                f"found {sorted(unique_labels)}."
            )

        if self.color_order not in {
            "RGB",
            "BGR",
        }:
            raise ValueError(
                f"Unsupported color_order={self.color_order}"
            )

    def __len__(
        self,
    ):
        return len(
            self.labels
        )

    def load_frames(
        self,
        source_idx: int,
    ) -> Tuple[np.ndarray, int]:
        frames = np.asarray(
            self.features[
                source_idx
            ],
            dtype=np.float32,
        )

        label = int(
            self.labels[
                source_idx
            ]
        )

        if not np.isfinite(
            frames
        ).all():
            raise ValueError(
                f"{self.dataset_name}, idx={source_idx}: NaN/Inf."
            )

        frame_min = float(
            frames.min()
        )

        frame_max = float(
            frames.max()
        )

        if (
            AUTO_SCALE_LEGACY_CACHE
            and frame_min >= 0.0
            and frame_max > 1.5
        ):
            if (
                frame_max
                <= 255.0
                + 1e-3
            ):
                frames = (
                    frames
                    / 255.0
                )

            else:
                raise ValueError(
                    f"{self.dataset_name}: invalid cache range "
                    f"[{frame_min},{frame_max}]."
                )

        if (
            float(
                frames.min()
            )
            < -1e-4
            or float(
                frames.max()
            )
            > 1.0001
        ):
            raise ValueError(
                f"{self.dataset_name}: cache must be [0,1]."
            )

        if self.color_order == "BGR":
            frames = frames[
                ...,
                [
                    2,
                    1,
                    0,
                ],
            ]

        return (
            frames,
            label,
        )


def selected_to_tensor(
    selected: np.ndarray,
) -> torch.Tensor:
    if (
        tuple(
            selected.shape
        )
        != (
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        )
    ):
        raise RuntimeError(
            f"Selected clip shape salah: {selected.shape}"
        )

    clip = torch.from_numpy(
        np.ascontiguousarray(
            selected
        )
    ).permute(
        0,
        3,
        1,
        2,
    )

    return (
        clip
        - IMAGENET_MEAN
    ) / IMAGENET_STD


class BiLSTMFinalDirectT16Dataset(
    HeldOut16CacheBase
):
    def __getitem__(
        self,
        source_idx: int,
    ):
        (
            frames,
            label,
        ) = self.load_frames(
            source_idx
        )

        clip = selected_to_tensor(
            frames
        )

        return (
            clip,
            torch.tensor(
                label,
                dtype=torch.long,
            ),
            int(
                source_idx
            ),
        )


class BiLSTMFinalFrameDiversityDataset(
    HeldOut16CacheBase
):
    def __init__(
        self,
        dataset_name: str,
        feature_path: Path,
        label_path: Path,
        color_order: str,
        k_unique: int,
        repeat_id: int,
    ):
        super().__init__(
            dataset_name=dataset_name,
            feature_path=feature_path,
            label_path=label_path,
            color_order=color_order,
        )

        self.k_unique = int(
            k_unique
        )

        self.repeat_id = int(
            repeat_id
        )

    def __getitem__(
        self,
        source_idx: int,
    ):
        (
            frames,
            label,
        ) = self.load_frames(
            source_idx
        )

        unique_indices = (
            stratified_unique_indices(
                dataset_name=self.dataset_name,
                source_idx=int(
                    source_idx
                ),
                k_unique=self.k_unique,
                repeat_id=self.repeat_id,
            )
        )

        expanded_indices = (
            expand_equal_weight_to_t16(
                unique_indices
            )
        )

        selected = frames[
            expanded_indices
        ]

        clip = selected_to_tensor(
            selected
        )

        return (
            clip,
            torch.tensor(
                label,
                dtype=torch.long,
            ),
            int(
                source_idx
            ),
            torch.from_numpy(
                unique_indices.copy()
            ),
            torch.from_numpy(
                expanded_indices.copy()
            ),
        )


def make_loader(
    dataset: Dataset,
):
    generator = torch.Generator()

    generator.manual_seed(
        SAMPLING_BASE_SEED
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type
            == "cuda"
        ),
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=(
            NUM_WORKERS
            > 0
        ),
    )


def load_paths_if_available(
    paths_path: Optional[Path],
    expected_n: int,
) -> Optional[List[str]]:
    if paths_path is None:
        return None

    path = Path(
        paths_path
    )

    if not path.exists():
        return None

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(
            file
        )

    paths = payload.get(
        "paths"
    )

    if not isinstance(
        paths,
        list,
    ):
        return None

    if (
        len(
            paths
        )
        != expected_n
    ):
        return None

    return [
        str(
            path
        )
        for path
        in paths
    ]

## 6. Resolve dan audit RWF, UCF, Bus Violence

In [ ]:
heldout_stores = {}

dataset_audit_rows = []

for (
    dataset_name,
    spec,
) in HELDOUT_SPECS.items():
    print(
        "\n"
        + "=" * 90
    )

    print(
        "DATASET:",
        dataset_name,
    )

    print(
        "=" * 90
    )

    store = resolve_heldout_store(
        dataset_name,
        spec,
    )

    dataset = BiLSTMFinalDirectT16Dataset(
        dataset_name=dataset_name,
        feature_path=store[
            "feature_path"
        ],
        label_path=store[
            "label_path"
        ],
        color_order=store[
            "color_order"
        ],
    )

    labels = np.asarray(
        dataset.labels,
        dtype=np.int64,
    )

    counts = np.bincount(
        labels,
        minlength=2,
    )

    heldout_stores[
        dataset_name
    ] = store

    dataset_audit_rows.append(
        {
            "dataset": dataset_name,
            "n_total": int(
                len(
                    dataset
                )
            ),
            "n_nonviolence": int(
                counts[
                    0
                ]
            ),
            "n_violence": int(
                counts[
                    1
                ]
            ),
            "cache_source": store[
                "cache_source"
            ],
            "color_order": store[
                "color_order"
            ],
            "feature_path": str(
                store[
                    "feature_path"
                ]
            ),
            "label_path": str(
                store[
                    "label_path"
                ]
            ),
        }
    )

    print(
        f"✅ {dataset_name}: "
        f"N={len(dataset)}, "
        f"NV={counts[0]}, "
        f"V={counts[1]}, "
        f"cache={store['cache_source']}"
    )

dataset_audit_df = pd.DataFrame(
    dataset_audit_rows
)

dataset_audit_df.to_csv(
    EVAL_OUTPUT_DIR
    / "heldout_dataset_audit.csv",
    index=False,
)

display(
    dataset_audit_df
)


DATASET: RWF
✅ RWF: legacy cache (BGR→RGB saat loading)
✅ RWF: N=1998, NV=1000, V=998, cache=legacy

DATASET: UCF
✅ UCF: RGB-v2 cache
✅ UCF: N=100, NV=50, V=50, cache=rgb64_v2

DATASET: BV
✅ BV: RGB-v2 cache
✅ BV: N=1400, NV=700, V=700, cache=rgb64_v2


,dataset,n_total,n_nonviolence,n_violence,cache_source,color_order,feature_path,label_path
0,RWF,1998,1000,998,legacy,BGR,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...
1,UCF,100,50,50,rgb64_v2,RGB,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...
2,BV,1400,700,700,rgb64_v2,RGB,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...


## 7. Metrics dan per-clip prediction export

Setiap prediction row menyimpan:
- checkpoint seed;
- dataset;
- K;
- repeat;
- sample key;
- distinct temporal indices;
- expanded T=16 indices;
- label;
- prediction;
- correctness;
- class probabilities.

Untuk mencegah error `NameError: prediction_df is not defined`, notebook ini
**tidak menggunakan variabel bernama `prediction_df` sama sekali**. Nama yang
dipakai eksplisit:
- `direct_predictions_df`
- `diversity_predictions_df`
- `condition_predictions_df`

In [ ]:
def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
) -> Dict[str, float]:
    (
        precision_cls,
        recall_cls,
        f1_cls,
        support,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[
            0,
            1,
        ],
        zero_division=0,
    )

    result = {
        "n": int(
            len(
                y_true
            )
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),

        "precision_nonviolence": float(
            precision_cls[
                0
            ]
        ),

        "recall_nonviolence": float(
            recall_cls[
                0
            ]
        ),

        "f1_nonviolence": float(
            f1_cls[
                0
            ]
        ),

        "precision_violence": float(
            precision_cls[
                1
            ]
        ),

        "recall_violence": float(
            recall_cls[
                1
            ]
        ),

        "f1_violence": float(
            f1_cls[
                1
            ]
        ),
    }

    if (
        len(
            np.unique(
                y_true
            )
        )
        == 2
    ):
        result[
            "roc_auc"
        ] = float(
            roc_auc_score(
                y_true,
                p_violence,
            )
        )

        result[
            "average_precision"
        ] = float(
            average_precision_score(
                y_true,
                p_violence,
            )
        )

    else:
        result[
            "roc_auc"
        ] = float(
            "nan"
        )

        result[
            "average_precision"
        ] = float(
            "nan"
        )

    return result


@torch.inference_mode()
def run_direct_loader(
    loader: DataLoader,
):
    model.eval()

    all_indices = []
    all_labels = []
    all_predictions = []
    all_probabilities = []

    for (
        clips,
        labels,
        indices,
    ) in tqdm(
        loader,
        leave=False,
        desc="Direct T16",
    ):
        clips = clips.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(
            clips
        )

        assert_finite_tensor(
            "direct_logits",
            logits,
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        predictions = torch.argmax(
            probabilities,
            dim=1,
        )

        all_indices.extend(
            np.asarray(
                indices
            ).tolist()
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

    return (
        np.asarray(
            all_indices,
            dtype=np.int64,
        ),
        np.asarray(
            all_labels,
            dtype=np.int64,
        ),
        np.asarray(
            all_predictions,
            dtype=np.int64,
        ),
        np.asarray(
            all_probabilities,
            dtype=np.float64,
        ),
    )


@torch.inference_mode()
def run_diversity_loader(
    loader: DataLoader,
):
    model.eval()

    all_indices = []
    all_labels = []
    all_predictions = []
    all_probabilities = []

    all_unique_indices = []
    all_expanded_indices = []

    for (
        clips,
        labels,
        indices,
        unique_indices,
        expanded_indices,
    ) in tqdm(
        loader,
        leave=False,
        desc="Frame diversity",
    ):
        clips = clips.to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(
            clips
        )

        assert_finite_tensor(
            "diversity_logits",
            logits,
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        predictions = torch.argmax(
            probabilities,
            dim=1,
        )

        all_indices.extend(
            np.asarray(
                indices
            ).tolist()
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

        all_unique_indices.extend(
            unique_indices.numpy().tolist()
        )

        all_expanded_indices.extend(
            expanded_indices.numpy().tolist()
        )

    return (
        np.asarray(
            all_indices,
            dtype=np.int64,
        ),
        np.asarray(
            all_labels,
            dtype=np.int64,
        ),
        np.asarray(
            all_predictions,
            dtype=np.int64,
        ),
        np.asarray(
            all_probabilities,
            dtype=np.float64,
        ),
        all_unique_indices,
        all_expanded_indices,
    )


def resolve_sample_keys(
    dataset_name: str,
    store: Dict,
    indices: np.ndarray,
):
    source_paths = load_paths_if_available(
        store.get(
            "paths_path"
        ),
        expected_n=len(
            indices
        ),
    )

    if source_paths is None:
        sample_keys = [
            f"{dataset_name}::{int(index)}"
            for index
            in indices
        ]

        video_paths = [
            ""
            for _
            in indices
        ]

    else:
        video_paths = [
            source_paths[
                int(
                    index
                )
            ]
            for index
            in indices
        ]

        sample_keys = [
            Path(
                path
            ).as_posix()
            for path
            in video_paths
        ]

    return (
        sample_keys,
        video_paths,
    )


def evaluate_direct_condition(
    dataset_name: str,
    store: Dict,
):
    dataset = BiLSTMFinalDirectT16Dataset(
        dataset_name=dataset_name,
        feature_path=store[
            "feature_path"
        ],
        label_path=store[
            "label_path"
        ],
        color_order=store[
            "color_order"
        ],
    )

    loader = make_loader(
        dataset
    )

    (
        indices,
        y_true,
        y_pred,
        probabilities,
    ) = run_direct_loader(
        loader
    )

    (
        sample_keys,
        video_paths,
    ) = resolve_sample_keys(
        dataset_name,
        store,
        indices,
    )

    metrics = compute_metrics(
        y_true,
        y_pred,
        probabilities[
            :,
            1
        ],
    )

    direct_predictions_df = pd.DataFrame(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "condition": "direct_t16",
            "k_unique": 16,
            "repeat": 0,
            "sample_index": indices,
            "sample_key": sample_keys,
            "video_path": video_paths,
            "y_true": y_true,
            "y_pred": y_pred,
            "is_correct": (
                y_true
                == y_pred
            ).astype(
                np.int64
            ),
            "p_nonviolence": probabilities[
                :,
                0
            ],
            "p_violence": probabilities[
                :,
                1
            ],
        }
    )

    if direct_predictions_df[
        "sample_key"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate sample_key: {dataset_name} direct."
        )

    return (
        metrics,
        direct_predictions_df,
    )


def evaluate_diversity_condition(
    dataset_name: str,
    store: Dict,
    k_unique: int,
    repeat_id: int,
):
    dataset = BiLSTMFinalFrameDiversityDataset(
        dataset_name=dataset_name,
        feature_path=store[
            "feature_path"
        ],
        label_path=store[
            "label_path"
        ],
        color_order=store[
            "color_order"
        ],
        k_unique=k_unique,
        repeat_id=repeat_id,
    )

    loader = make_loader(
        dataset
    )

    (
        indices,
        y_true,
        y_pred,
        probabilities,
        unique_indices,
        expanded_indices,
    ) = run_diversity_loader(
        loader
    )

    (
        sample_keys,
        video_paths,
    ) = resolve_sample_keys(
        dataset_name,
        store,
        indices,
    )

    unique_index_strings = [
        json.dumps(
            positions
        )
        for positions
        in unique_indices
    ]

    expanded_index_strings = [
        json.dumps(
            positions
        )
        for positions
        in expanded_indices
    ]

    metrics = compute_metrics(
        y_true,
        y_pred,
        probabilities[
            :,
            1
        ],
    )

    diversity_predictions_df = pd.DataFrame(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "condition": "frame_diversity",
            "k_unique": k_unique,
            "repeat": repeat_id,
            "sample_index": indices,
            "sample_key": sample_keys,
            "video_path": video_paths,
            "distinct_indices_0based": unique_index_strings,
            "expanded_indices_0based": expanded_index_strings,
            "y_true": y_true,
            "y_pred": y_pred,
            "is_correct": (
                y_true
                == y_pred
            ).astype(
                np.int64
            ),
            "p_nonviolence": probabilities[
                :,
                0
            ],
            "p_violence": probabilities[
                :,
                1
            ],
        }
    )

    if diversity_predictions_df[
        "sample_key"
    ].duplicated().any():
        raise RuntimeError(
            f"Duplicate sample_key: "
            f"{dataset_name}, K={k_unique}, repeat={repeat_id}."
        )

    return (
        metrics,
        diversity_predictions_df,
    )

## 8. K=16 reproduction gate

Sebelum K=1/2/4/8 dijalankan:

```text
Direct T=16
vs
FrameDiversity K=16
```

harus sama clip-by-clip.

Gate memeriksa:
- jumlah sample;
- `sample_key`;
- label;
- `p_nonviolence`;
- `p_violence`;
- Accuracy;
- Balanced Accuracy;
- Macro-F1;
- ROC-AUC;
- AP.

Jika gate gagal, notebook berhenti.

In [ ]:
direct_metric_rows = []

direct_prediction_frames = []

k16_metric_rows = []

k16_prediction_frames = []

gate_rows = []

for (
    dataset_name,
    store,
) in heldout_stores.items():
    print(
        "\n"
        + "=" * 90
    )

    print(
        f"K=16 REPRODUCTION GATE | {dataset_name}"
    )

    print(
        "=" * 90
    )

    (
        direct_metrics,
        direct_predictions_df,
    ) = evaluate_direct_condition(
        dataset_name,
        store,
    )

    (
        k16_metrics,
        k16_predictions_df,
    ) = evaluate_diversity_condition(
        dataset_name=dataset_name,
        store=store,
        k_unique=16,
        repeat_id=0,
    )

    merged = direct_predictions_df[
        [
            "sample_key",
            "y_true",
            "p_nonviolence",
            "p_violence",
        ]
    ].merge(
        k16_predictions_df[
            [
                "sample_key",
                "y_true",
                "p_nonviolence",
                "p_violence",
            ]
        ],
        on="sample_key",
        how="inner",
        suffixes=(
            "_direct",
            "_k16",
        ),
        validate="one_to_one",
    )

    if (
        len(
            merged
        )
        != len(
            direct_predictions_df
        )
    ):
        raise RuntimeError(
            f"{dataset_name}: direct/K16 sample-count mismatch."
        )

    if not np.array_equal(
        merged[
            "y_true_direct"
        ].to_numpy(),
        merged[
            "y_true_k16"
        ].to_numpy(),
    ):
        raise RuntimeError(
            f"{dataset_name}: direct/K16 label mismatch."
        )

    max_probability_delta = float(
        np.max(
            np.abs(
                merged[
                    [
                        "p_nonviolence_direct",
                        "p_violence_direct",
                    ]
                ].to_numpy()
                -
                merged[
                    [
                        "p_nonviolence_k16",
                        "p_violence_k16",
                    ]
                ].to_numpy()
            )
        )
    )

    metric_deltas = {}

    for metric_name in [
        "accuracy",
        "balanced_accuracy",
        "f1_macro",
        "roc_auc",
        "average_precision",
    ]:
        metric_deltas[
            metric_name
        ] = float(
            k16_metrics[
                metric_name
            ]
            -
            direct_metrics[
                metric_name
            ]
        )

    finite_metric_deltas = [
        abs(
            value
        )
        for value
        in metric_deltas.values()
        if np.isfinite(
            value
        )
    ]

    max_metric_delta = (
        float(
            max(
                finite_metric_deltas
            )
        )
        if finite_metric_deltas
        else 0.0
    )

    gate_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            "max_abs_probability_delta": max_probability_delta,
            "max_abs_metric_delta": max_metric_delta,
            **{
                f"delta_{key}": value
                for key, value
                in metric_deltas.items()
            },
        }
    )

    if (
        max_probability_delta
        > K16_PROBABILITY_ATOL
    ):
        raise RuntimeError(
            f"{dataset_name}: K16 probability reproduction failed: "
            f"{max_probability_delta} > {K16_PROBABILITY_ATOL}"
        )

    if (
        max_metric_delta
        > K16_METRIC_ATOL
    ):
        raise RuntimeError(
            f"{dataset_name}: K16 metric reproduction failed: "
            f"{max_metric_delta} > {K16_METRIC_ATOL}"
        )

    direct_metric_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "dataset": dataset_name,
            **direct_metrics,
        }
    )

    direct_prediction_frames.append(
        direct_predictions_df
    )

    k16_metric_rows.append(
        {
            "checkpoint_seed": RUN_SEED,
            "k_unique": 16,
            "repeat": 0,
            "dataset": dataset_name,
            **k16_metrics,
        }
    )

    k16_prediction_frames.append(
        k16_predictions_df
    )

    print(
        f"✅ {dataset_name}: "
        f"max Δprob={max_probability_delta:.3e}, "
        f"max Δmetric={max_metric_delta:.3e}"
    )

gate_df = pd.DataFrame(
    gate_rows
)

gate_df.to_csv(
    EVAL_OUTPUT_DIR
    / f"bilstm_final_K16_direct_reproduction_gate_seed{RUN_SEED}.csv",
    index=False,
)

pd.DataFrame(
    direct_metric_rows
).to_csv(
    EVAL_OUTPUT_DIR
    / f"bilstm_final_direct_T16_metrics_seed{RUN_SEED}.csv",
    index=False,
)

pd.concat(
    direct_prediction_frames,
    ignore_index=True,
).to_csv(
    EVAL_OUTPUT_DIR
    / f"bilstm_final_direct_T16_predictions_seed{RUN_SEED}.csv",
    index=False,
)

display(
    gate_df
)

print(
    "\n✅ Semua K=16 reproduction gates PASS"
)


K=16 REPRODUCTION GATE | RWF


Direct T16:   0%|          | 0/500 [00:00<?, ?it/s]

Frame diversity:   0%|          | 0/500 [00:00<?, ?it/s]

✅ RWF: max Δprob=0.000e+00, max Δmetric=0.000e+00

K=16 REPRODUCTION GATE | UCF


Direct T16:   0%|          | 0/25 [00:00<?, ?it/s]

...

Direct T16:   0%|          | 0/350 [00:00<?, ?it/s]

Frame diversity:   0%|          | 0/350 [00:00<?, ?it/s]

✅ BV: max Δprob=0.000e+00, max Δmetric=0.000e+00


,checkpoint_seed,dataset,max_abs_probability_delta,max_abs_metric_delta,delta_accuracy,delta_balanced_accuracy,delta_f1_macro,delta_roc_auc,delta_average_precision
0,0,RWF,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,UCF,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,BV,0.0,0.0,0.0,0.0,0.0,0.0,0.0



✅ Semua K=16 reproduction gates PASS


## 9. Jalankan K=1/2/4/8 × 10 repeats + K=16 reference

In [ ]:
metric_frames = [
    pd.DataFrame(
        k16_metric_rows
    )
]

prediction_frames = [
    pd.concat(
        k16_prediction_frames,
        ignore_index=True,
    )
]

for k_unique in [
    1,
    2,
    4,
    8,
]:
    for repeat_id in range(
        NUM_RANDOM_REPEATS
    ):
        print(
            f"\nseed={RUN_SEED} | "
            f"K={k_unique} | "
            f"repeat={repeat_id}"
        )

        condition_metric_rows = []

        condition_prediction_frames = []

        for (
            dataset_name,
            store,
        ) in heldout_stores.items():
            (
                metrics,
                condition_predictions_df,
            ) = evaluate_diversity_condition(
                dataset_name=dataset_name,
                store=store,
                k_unique=k_unique,
                repeat_id=repeat_id,
            )

            condition_metric_rows.append(
                {
                    "checkpoint_seed": RUN_SEED,
                    "k_unique": k_unique,
                    "repeat": repeat_id,
                    "dataset": dataset_name,
                    **metrics,
                }
            )

            condition_prediction_frames.append(
                condition_predictions_df
            )

        metric_frames.append(
            pd.DataFrame(
                condition_metric_rows
            )
        )

        prediction_frames.append(
            pd.concat(
                condition_prediction_frames,
                ignore_index=True,
            )
        )

all_metrics = pd.concat(
    metric_frames,
    ignore_index=True,
)

all_predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)

metrics_path = (
    EVAL_OUTPUT_DIR
    / f"bilstm_final_frame_diversity_all_repeats_seed{RUN_SEED}.csv"
)

predictions_path = (
    EVAL_OUTPUT_DIR
    / f"bilstm_final_frame_diversity_predictions_seed{RUN_SEED}.csv"
)

all_metrics.to_csv(
    metrics_path,
    index=False,
)

all_predictions.to_csv(
    predictions_path,
    index=False,
)

print(
    "\n✅ Semua held-out BiLSTM-Final frame-diversity conditions selesai."
)

print(
    "Metrics    :",
    metrics_path,
)

print(
    "Predictions:",
    predictions_path,
)


seed=0 | K=1 | repeat=0


Frame diversity:   0%|          | 0/500 [00:00<?, ?it/s]

Frame diversity:   0%|          | 0/25 [00:00<?, ?it/s]

Frame diversity:   0%|          | 0/350 [00:00<?, ?it/s]


seed=0 | K=1 | repeat=1


...


seed=0 | K=8 | repeat=9


Frame diversity:   0%|          | 0/500 [00:00<?, ?it/s]

Frame diversity:   0%|          | 0/25 [00:00<?, ?it/s]

Frame diversity:   0%|          | 0/350 [00:00<?, ?it/s]


✅ Semua held-out BiLSTM-Final frame-diversity conditions selesai.
Metrics    : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_FrameDiversity/seed_0/bilstm_final_frame_diversity_all_repeats_seed0.csv
Predictions: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_FrameDiversity/seed_0/bilstm_final_frame_diversity_predictions_seed0.csv


## 10. Summary mean ± temporal-sampling std

Untuk K=1/2/4/8, `std_sampling` berasal dari **10 temporal-sampling repeats**,
bukan dari checkpoint seeds.

K=16 reference hanya satu deterministic condition dan memiliki
`std_sampling = 0`.

In [ ]:
METRIC_COLUMNS = [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "balanced_accuracy",
    "f1_macro",
    "roc_auc",
    "average_precision",
    "precision_nonviolence",
    "recall_nonviolence",
    "f1_nonviolence",
    "precision_violence",
    "recall_violence",
    "f1_violence",
]

summary_rows = []

for (
    dataset_name,
    k_unique,
), group in all_metrics.groupby(
    [
        "dataset",
        "k_unique",
    ]
):
    row = {
        "checkpoint_seed": RUN_SEED,
        "dataset": dataset_name,
        "k_unique": int(
            k_unique
        ),
        "num_sampling_repeats": int(
            len(
                group
            )
        ),
    }

    for metric_name in METRIC_COLUMNS:
        values = group[
            metric_name
        ].astype(
            float
        )

        row[
            f"{metric_name}_mean"
        ] = float(
            values.mean()
        )

        row[
            f"{metric_name}_std_sampling"
        ] = float(
            values.std(
                ddof=1
            )
            if len(
                values
            )
            > 1
            else 0.0
        )

        row[
            f"{metric_name}_min"
        ] = float(
            values.min()
        )

        row[
            f"{metric_name}_max"
        ] = float(
            values.max()
        )

    summary_rows.append(
        row
    )

summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    [
        "dataset",
        "k_unique",
    ]
)

summary_path = (
    EVAL_OUTPUT_DIR
    / f"bilstm_final_frame_diversity_summary_seed{RUN_SEED}.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
)

display(
    summary_df[
        [
            "checkpoint_seed",
            "dataset",
            "k_unique",
            "num_sampling_repeats",
            "average_precision_mean",
            "average_precision_std_sampling",
            "roc_auc_mean",
            "roc_auc_std_sampling",
            "f1_macro_mean",
            "f1_macro_std_sampling",
        ]
    ].round(
        5
    )
)

print(
    "Saved:",
    summary_path,
)

,checkpoint_seed,dataset,k_unique,num_sampling_repeats,average_precision_mean,average_precision_std_sampling,roc_auc_mean,roc_auc_std_sampling,f1_macro_mean,f1_macro_std_sampling
0,0,BV,1,10,0.49735,0.00655,0.49784,0.00678,0.48988,0.01006
1,0,BV,2,10,0.49315,0.00494,0.49112,0.00533,0.48037,0.00525
2,0,BV,4,10,0.49006,0.00421,0.49186,0.00485,0.48025,0.00677
3,0,BV,8,10,0.48829,0.00185,0.49037,0.00232,0.48089,0.00269
4,0,BV,16,1,0.49025,0.00000,0.49285,0.00000,0.48302,0.00000
5,0,RWF,1,10,0.61434,0.00842,0.61753,0.00758,0.58922,0.00666
6,0,RWF,2,10,0.62036,0.00626,0.62758,0.00516,0.59614,0.00681
7,0,RWF,4,10,0.62456,0.00404,0.63400,0.00388,0.60198,0.00428
8,0,RWF,8,10,0.62822,0.00198,0.64078,0.00094,0.60415,0.00488
9,0,RWF,16,1,0.63001,0.00000,0.64300,0.00000,0.61192,0.00000


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_FrameDiversity/seed_0/bilstm_final_frame_diversity_summary_seed0.csv


## 11. Delta terhadap K=16

Primary paper metric: AP.

```text
ΔAP      = AP(K=16) - AP(K)
ΔAUC     = AUC(K=16) - AUC(K)
ΔMacroF1 = MacroF1(K=16) - MacroF1(K)
```

Kolom `delta_*_pp` sudah dalam **poin persentase**.

In [ ]:
k16_reference = (
    summary_df[
        summary_df[
            "k_unique"
        ]
        == 16
    ][
        [
            "dataset",
            "average_precision_mean",
            "roc_auc_mean",
            "f1_macro_mean",
        ]
    ]
    .rename(
        columns={
            "average_precision_mean": "ap_k16",
            "roc_auc_mean": "auc_k16",
            "f1_macro_mean": "f1_k16",
        }
    )
)

delta_df = summary_df.merge(
    k16_reference,
    on="dataset",
    how="left",
)

delta_df[
    "delta_ap_k16_minus_k"
] = (
    delta_df[
        "ap_k16"
    ]
    - delta_df[
        "average_precision_mean"
    ]
)

delta_df[
    "delta_auc_k16_minus_k"
] = (
    delta_df[
        "auc_k16"
    ]
    - delta_df[
        "roc_auc_mean"
    ]
)

delta_df[
    "delta_f1_k16_minus_k"
] = (
    delta_df[
        "f1_k16"
    ]
    - delta_df[
        "f1_macro_mean"
    ]
)

delta_df[
    "delta_ap_pp"
] = (
    100.0
    * delta_df[
        "delta_ap_k16_minus_k"
    ]
)

delta_df[
    "delta_auc_pp"
] = (
    100.0
    * delta_df[
        "delta_auc_k16_minus_k"
    ]
)

delta_df[
    "delta_f1_pp"
] = (
    100.0
    * delta_df[
        "delta_f1_k16_minus_k"
    ]
)

delta_path = (
    EVAL_OUTPUT_DIR
    / f"bilstm_final_frame_diversity_delta_seed{RUN_SEED}.csv"
)

delta_df.to_csv(
    delta_path,
    index=False,
)

display(
    delta_df[
        [
            "dataset",
            "k_unique",
            "average_precision_mean",
            "delta_ap_pp",
            "roc_auc_mean",
            "delta_auc_pp",
            "f1_macro_mean",
            "delta_f1_pp",
        ]
    ].round(
        4
    )
)

print(
    "Saved:",
    delta_path,
)

,dataset,k_unique,average_precision_mean,delta_ap_pp,roc_auc_mean,delta_auc_pp,f1_macro_mean,delta_f1_pp
0,BV,1,0.4974,-0.7105,0.4978,-0.4987,0.4899,-0.6862
1,BV,2,0.4932,-0.2907,0.4911,0.1731,0.4804,0.2648
2,BV,4,0.4901,0.0181,0.4919,0.0991,0.4802,0.2774
3,BV,8,0.4883,0.1960,0.4904,0.2485,0.4809,0.2129
4,BV,16,0.4902,0.0000,0.4929,0.0000,0.4830,0.0000
5,RWF,1,0.6143,1.5676,0.6175,2.5465,0.5892,2.2701
6,RWF,2,0.6204,0.9655,0.6276,1.5414,0.5961,1.5776
7,RWF,4,0.6246,0.5455,0.6340,0.8997,0.6020,0.9939
8,RWF,8,0.6282,0.1792,0.6408,0.2218,0.6041,0.7770
9,RWF,16,0.6300,0.0000,0.6430,0.0000,0.6119,0.0000


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/BiLSTMFinal_FrameDiversity/seed_0/bilstm_final_frame_diversity_delta_seed0.csv


## 12. Protocol manifest

In [ ]:
protocol_audit = {
    "task": "heldout_frame_diversity_evaluation_only",
    "model": "EffTempNet BiLSTM-Final",
    "variant": VARIANT,
    "checkpoint_seed": RUN_SEED,
    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),
    "checkpoint_epoch": checkpoint.get(
        "epoch"
    ),
    "training_performed": False,
    "fine_tuning_performed": False,
    "strict_checkpoint_loading": True,
    "sequence_length": SEQUENCE_LENGTH,
    "lstm_units": LSTM_UNITS,
    "bidirectional": True,
    "temporal_aggregation": (
        "concat final forward and backward hidden states"
    ),
    "uses_temporal_attention": False,
    "tested_distinct_frame_counts": K_VALUES,
    "random_repeats_for_k_1_2_4_8": NUM_RANDOM_REPEATS,
    "sampling_base_seed": SAMPLING_BASE_SEED,
    "checkpoint_seed_excluded_from_sampling_generation": True,
    "sampling_identity_inputs": [
        "sampling_base_seed",
        "repeat_id",
        "dataset_name_crc32",
        "sample_index",
        "k_unique",
    ],
    "sampling_strategy": (
        "one reproducible random temporal position from each "
        "equal-width temporal bin on the original T=16 grid"
    ),
    "expansion_strategy": (
        "selected distinct temporal positions repeated in chronological "
        "blocks with equal weight until T=16"
    ),
    "k16_direct_reproduction_gate": True,
    "k16_probability_atol": K16_PROBABILITY_ATOL,
    "k16_metric_atol": K16_METRIC_ATOL,
    "heldout_datasets": list(
        HELDOUT_SPECS.keys()
    ),
    "no_cross_dataset_pooling": True,
    "primary_metric": "average_precision",
    "important_k1_note": (
        "K=1 is not the separately trained SingleFrame model. "
        "Checkpoint and model interface remain T=16."
    ),
    "interpretation_limit": (
        "Inference-time sensitivity to temporal diversity/redundancy "
        "at fixed T=16. Because BiLSTM is sequential, reducing K also "
        "changes temporal repetition and transition density. "
        "This is not a pure causal isolation of evidence quantity "
        "and is not a compute-saving test."
    ),
}

manifest_path = (
    EVAL_OUTPUT_DIR
    / f"bilstm_final_frame_diversity_protocol_seed{RUN_SEED}.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        protocol_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        protocol_audit,
        indent=2,
    )
)

print(
    "\nOutput:",
    EVAL_OUTPUT_DIR,
)

{
  "task": "heldout_frame_diversity_evaluation_only",
  "model": "EffTempNet BiLSTM-Final",
  "variant": "bilstm_final",
  "checkpoint_seed": 0,
  "checkpoint_path": "/content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/EffTempNet_BiLSTMFinal_NoLS_Seed0_v1/bilstm_final/seed_0/best.pt",
  "checkpoint_epoch": 14,
  "training_performed": false,
  "fine_tuning_performed": false,
  "strict_checkpoint_loading": true,
  "sequence_length": 16,
  "lstm_units": 32,
  "bidirectional": true,
  "temporal_aggregation": "concat final forward and backward hidden states",
  "uses_temporal_attention": false,
  "tested_distinct_frame_counts": [
    1,
    2,
    4,
    8,
    16
  ],
  "random_repeats_for_k_1_2_4_8": 10,
  "sampling_base_seed": 20260810,
  "checkpoint_seed_excluded_from_sampling_generation": true,
  "sampling_identity_inputs": [
    "sampling_base_seed",
    "repeat_id",
    "dataset_name_crc32",
    "sample_index",
    "k_unique"
  ],
  "sampling_strategy": "one reproducible 

# Seed berikutnya

Untuk checkpoint berikutnya, **cukup ubah**:

```python
RUN_SEED = 0
```

menjadi:

```python
RUN_SEED = 42
```

kemudian:

```python
RUN_SEED = 123
```

Jangan ubah:

```python
SEQUENCE_LENGTH = 16
LSTM_UNITS = 32
K_VALUES = [1,2,4,8,16]
NUM_RANDOM_REPEATS = 10
SAMPLING_BASE_SEED = 20260810
BATCH_SIZE = 4
```

Output otomatis dipisahkan:

```text
HeldOut_Evaluation/
└── BiLSTMFinal_FrameDiversity/
    ├── seed_0/
    ├── seed_42/
    └── seed_123/
```

File utama:
- `bilstm_final_frame_diversity_all_repeats_seedX.csv`
- `bilstm_final_frame_diversity_predictions_seedX.csv`
- `bilstm_final_frame_diversity_summary_seedX.csv`
- `bilstm_final_frame_diversity_delta_seedX.csv`
- `bilstm_final_K16_direct_reproduction_gate_seedX.csv`
- `bilstm_final_direct_T16_metrics_seedX.csv`
- `bilstm_final_direct_T16_predictions_seedX.csv`
- `heldout_dataset_audit.csv`
- `frame_selection_audit_examples.json`
- `bilstm_final_frame_diversity_protocol_seedX.json`